[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/state-reducers.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239428-lesson-2-state-reducers)

# 状态 Reducer

## 回顾

我们介绍了定义 LangGraph 状态 Schema 的几种不同方式，包括 `TypedDict`、`Pydantic` 或 `Dataclasses`。

## 目标

现在，我们将深入探讨 reducer，它指定了如何对状态 Schema 中特定的键/通道执行状态更新。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph

## 默认覆盖状态

让我们使用 `TypedDict` 作为状态 Schema。

In [ ]:
from typing_extensions import TypedDict
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    foo: int

def node_1(state):
    print("---Node 1---")
    return {"foo": state['foo'] + 1}

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"foo" : 1})

让我们看一下状态更新，`return {"foo": state['foo'] + 1}`。

如前所述，默认情况下 LangGraph 不知道更新状态的优先方式。

因此，它会在 `node_1` 中直接覆盖 `foo` 的值：

```
return {"foo": state['foo'] + 1}
```

如果我们传入 `{'foo': 1}` 作为输入，图返回的状态是 `{'foo': 2}`。

## 分支

让我们看一个节点出现分支的情况。

In [ ]:
class State(TypedDict):
    foo: int

def node_1(state):
    print("---Node 1---")
    return {"foo": state['foo'] + 1}

def node_2(state):
    print("---Node 2---")
    return {"foo": state['foo'] + 1}

def node_3(state):
    print("---Node 3---")
    return {"foo": state['foo'] + 1}

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langgraph.errors import InvalidUpdateError
try:
    graph.invoke({"foo" : 1})
except InvalidUpdateError as e:
    print(f"InvalidUpdateError occurred: {e}")


我们看到一个问题！

节点 1 分支到节点 2 和节点 3。

节点 2 和节点 3 并行运行，这意味着它们在图的同一个步骤中运行。

它们都试图*在同一个步骤中*覆盖状态。

这对图来说是模棱两可的！它应该保留哪个状态？

## Reducer

[Reducer](https://docs.langchain.com/oss/python/langgraph/graph-api/#reducers) 为我们提供了一种解决此问题的通用方法。

它们指定了如何执行更新。

我们可以使用 `Annotated` 类型来指定一个 reducer 函数。

例如，在本例中，让我们追加从每个节点返回的值，而不是覆盖它们。

我们只需要一个能执行此操作的 reducer：`operator.add` 是 Python 内置 operator 模块中的一个函数。

当 `operator.add` 应用于列表时，它会执行列表拼接。

In [ ]:
from operator import add
from typing import Annotated

class State(TypedDict):
    foo: Annotated[list[int], add]

def node_1(state):
    print("---Node 1---")
    return {"foo": [state['foo'][0] + 1]}

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
graph.invoke({"foo" : [1]})

现在，我们的状态键 `foo` 是一个列表。

这个 `operator.add` reducer 函数会将每个节点的更新追加到这个列表中。

In [ ]:
def node_1(state):
    print("---Node 1---")
    return {"foo": [state['foo'][-1] + 1]}

def node_2(state):
    print("---Node 2---")
    return {"foo": [state['foo'][-1] + 1]}

def node_3(state):
    print("---Node 3---")
    return {"foo": [state['foo'][-1] + 1]}

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_1", "node_3")
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

我们可以看到节点 2 和节点 3 中的更新是并发执行的，因为它们处于同一个步骤中。

In [ ]:
graph.invoke({"foo" : [1]})

现在，让我们看看如果将 `None` 传递给 `foo` 会发生什么。

我们看到一个错误，因为我们的 reducer `operator.add` 试图将 `NoneType` 作为输入与 `node_1` 中的列表进行拼接。

In [ ]:
try:
    graph.invoke({"foo" : None})
except TypeError as e:
    print(f"TypeError occurred: {e}")

## 自定义 Reducer

为了解决此类情况，[我们也可以定义自定义 reducer](https://docs.langchain.com/oss/python/langgraph/use-graph-api#process-state-updates-with-reducers)。

例如，让我们定义自定义 reducer 逻辑来合并列表，并处理输入中任意一个或两个都可能为 `None` 的情况。

In [ ]:
def reduce_list(left: list | None, right: list | None) -> list:
    """安全地合并两个列表，处理任意一个或两个输入都可能为 None 的情况。

    Args:
        left (list | None): 要合并的第一个列表，或 None。
        right (list | None): 要合并的第二个列表，或 None。

    Returns:
        list: 一个包含两个输入列表中所有元素的新列表。
               如果某个输入为 None，则将其视为空列表。
    """
    if not left:
        left = []
    if not right:
        right = []
    return left + right

class DefaultState(TypedDict):
    foo: Annotated[list[int], add]

class CustomReducerState(TypedDict):
    foo: Annotated[list[int], reduce_list]

在 `node_1` 中，我们追加值 2。

In [ ]:
def node_1(state):
    print("---Node 1---")
    return {"foo": [2]}

# 构建图
builder = StateGraph(DefaultState)
builder.add_node("node_1", node_1)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

try:
    print(graph.invoke({"foo" : None}))
except TypeError as e:
    print(f"TypeError occurred: {e}")

现在，试试我们的自定义 reducer。我们可以看到没有抛出错误。

In [ ]:
# 构建图
builder = StateGraph(CustomReducerState)
builder.add_node("node_1", node_1)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

try:
    print(graph.invoke({"foo" : None}))
except TypeError as e:
    print(f"TypeError occurred: {e}")

## Messages

在模块 1 中，我们展示了如何使用内置的 reducer `add_messages` 来处理状态中的消息。

我们还展示了 [`MessagesState` 是一个有用的快捷方式，如果你想处理消息的话](https://docs.langchain.com/oss/python/langgraph/use-graph-api#messagesstate)。

* `MessagesState` 有一个内置的 `messages` 键
* 它还有一个内置的 `add_messages` reducer 用于此键

这两种方式是等价的。

为简洁起见，我们将使用 `MessagesState` 类，通过 `from langgraph.graph import MessagesState`。

In [ ]:
from typing import Annotated
from langgraph.graph import MessagesState
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

# 定义一个自定义的 TypedDict，包含带 add_messages reducer 的消息列表
class CustomMessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    added_key_1: str
    added_key_2: str
    # 等等

# 使用 MessagesState，它包含了带有 add_messages reducer 的 messages 键
class ExtendedMessagesState(MessagesState):
    # 添加除了预置的 messages 之外可能需要的键
    added_key_1: str
    added_key_2: str
    # 等等

让我们再谈谈 `add_messages` reducer 的使用方式。

In [ ]:
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage, HumanMessage

# 初始状态
initial_messages = [AIMessage(content="Hello! How can I assist you?", name="Model"),
                    HumanMessage(content="I'm looking for information on marine biology.", name="Lance")
                   ]

# 要添加的新消息
new_message = AIMessage(content="Sure, I can help with that. What specifically are you interested in?", name="Model")

# 测试
add_messages(initial_messages , new_message)

我们可以看到 `add_messages` 允许我们将消息追加到状态中的 `messages` 键。

### 重写

让我们展示一些在使用 `add_messages` reducer 时的实用技巧。

如果我们传入一条与 `messages` 列表中现有消息具有相同 ID 的消息，它将被覆盖！

In [ ]:
# 初始状态
initial_messages = [AIMessage(content="Hello! How can I assist you?", name="Model", id="1"),
                    HumanMessage(content="I'm looking for information on marine biology.", name="Lance", id="2")
                   ]

# 要添加的新消息
new_message = HumanMessage(content="I'm looking for information on whales, specifically", name="Lance", id="2")

# 测试
add_messages(initial_messages , new_message)

### 删除

我们可以使用 [RemoveMessage](https://docs.langchain.com/oss/python/langgraph/add-memory#delete-messages) 来删除消息。

In [ ]:
from langchain_core.messages import RemoveMessage

# 消息列表
messages = [AIMessage("Hi.", name="Bot", id="1")]
messages.append(HumanMessage("Hi.", name="Lance", id="2"))
messages.append(AIMessage("So you said you were researching ocean mammals?", name="Bot", id="3"))
messages.append(HumanMessage("Yes, I know about whales. But what others should I learn about?", name="Lance", id="4"))

# 隔离要删除的消息
delete_messages = [RemoveMessage(id=m.id) for m in messages[:-2]]
print(delete_messages)

In [ ]:
add_messages(messages , delete_messages)

我们可以看到，`delete_messages` 中标明的消息 ID 1 和 2 被 reducer 删除了。

我们稍后会看到这个功能的实际应用。